# SOXL Backtest Policy Reimplementation v1

This notebook validates a current-harness deterministic reimplementation of the legacy non-qlib `soxl_backtest` rule policy. The validation uses existing local repo data only and compares the new implementation against the previously persisted legacy oracle artifacts.

## Acceptance Targets

- Recompute legacy-compatible features instead of using qlib approximations.
- Keep the legacy package as the oracle, not the implementation path.
- Match the 2026 YTD fresh-start return near `132.08%` and trade dates near `2026-03-30`, `2026-05-27`, `2026-06-05`.
- Match the June 2026 fresh-start return near `3.14%` and trade date near `2026-06-05`.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from scripts.export_legacy_soxl_policy_replay import write_legacy_soxl_policy_replay_outputs

OUTPUT_DIR = Path("runtime/production/card/soxl_backtest_policy_reimplementation_v1")
summary = write_legacy_soxl_policy_replay_outputs(OUTPUT_DIR)
metrics = pd.read_csv(OUTPUT_DIR / "legacy_comparison_metrics.csv")
timing = pd.read_csv(OUTPUT_DIR / "trade_timing_diagnostics.csv")
trades = pd.read_csv(OUTPUT_DIR / "new_policy_trades.csv")
daily = pd.read_csv(OUTPUT_DIR / "new_policy_daily.csv")

print(json.dumps({
    "summary": summary["artifacts"]["summary"],
    "metric_rows": len(metrics),
    "timing_rows": len(timing),
    "trade_rows": len(trades),
    "daily_rows": len(daily),
}, indent=2))

{
  "summary": "runtime/production/card/soxl_backtest_policy_reimplementation_v1/summary.json",
  "metric_rows": 2,
  "timing_rows": 4,
  "trade_rows": 4,
  "daily_rows": 116
}


## Return And Position Comparison

In [2]:
metrics

,window,legacy_window,start,end,new_return_pct,legacy_return_pct,return_delta_pct,new_end_position_pct,legacy_end_position_pct,end_position_delta_pct,new_trade_count,legacy_trade_count,trade_count_delta,new_max_drawdown_pct,legacy_max_drawdown_pct
0,ytd_2026,legacy_ytd_2026_fresh_start,2026-01-02,2026-06-09,132.077978,132.077978,0.000000e+00,15.91,15.91,0.0,3,3,0,13.554089,13.554089
1,june_2026,legacy_june_2026_fresh_start,2026-06-01,2026-06-09,3.138960,3.138960,4.440892e-15,29.94,29.94,0.0,1,1,0,1.528209,1.528209


## Trade Timing Diagnostics

In [3]:
timing

,window,sequence,new_date,legacy_date,date_delta_days,new_type,legacy_type,new_position_pct,legacy_position_pct
0,ytd_2026,1,2026-03-30,2026-03-30,0,buy,buy,29.98,29.98
1,ytd_2026,2,2026-05-27,2026-05-27,0,take_profit,take_profit,6.99,6.99
2,ytd_2026,3,2026-06-05,2026-06-05,0,buy,buy,15.91,15.91
3,june_2026,1,2026-06-05,2026-06-05,0,buy,buy,29.94,29.94


## New Policy Trades

In [4]:
trades[["window", "date", "type", "price", "position", "score", "buy_type", "market_condition", "buy_step", "gain"]]

,window,date,type,price,position,score,buy_type,market_condition,buy_step,gain
0,ytd_2026,2026-03-30,buy,40.619999,0.2998,-150.000000,cumulative,undervalued,0.3,NaN
1,ytd_2026,2026-05-27,take_profit,217.979996,0.0699,99.615509,NaN,NaN,NaN,436.632204
2,ytd_2026,2026-06-05,buy,182.540000,0.1591,-75.303575,daily,undervalued,0.1,NaN
3,june_2026,2026-06-05,buy,182.540000,0.2994,-83.939810,daily,undervalued,0.3,NaN


## Conclusion

- Implemented a deterministic current-harness replay of the non-qlib `soxl_backtest` feature and rule logic using existing local data only.
- YTD fresh-start window `2026-01-02` to `2026-06-09`: new policy return `132.07797795135497%` vs legacy `132.07797795135497%`, return delta `0.0pp`, trade dates exactly matched.
- June fresh-start window `2026-06-01` to `2026-06-09`: new policy return `3.1389600000000017%` vs legacy `3.1389599999999973%`, return delta approximately `0.0pp`, trade date exactly matched.
- Promotion decision: this is a parity/reimplementation pass, not a production-composer replacement. Next action is optional integration into dashboard/model replay discovery if this rule policy should become a first-class comparable strategy.